In [ ]:
import os
import subprocess

os.chdir('/workspace')

from ipywidgets import interact, ToggleButtons
from models.e2e_pipeline import graph_optimizer_e2e
from models.hardware import extract_hardware_characteristics, hardware_and_microbenchmarks
from models.prediction import analytical_prediction, sampling_based_prediction, convert_to_greenifier_input, select_worst_case
from code_generation.code_gen import generate_code
from data.graph_properties import extract_graph_properties
from beta_testing.plot_predictions import plot_annotated_dag
from demo.review2.utils import plot_runtimes, convert_output, get_error

RUNTIME = 0
ENERGY = 1

# Graph-Optimizer Demo
## End to end prediction and code generation

<img src="e2e.svg">

Example of a workflow

<img src='dag.svg'>

In [ ]:
workflow = [
    {
        "id": 0,
        "name": "pr",
        "dependencies": [],
        "inputs": [{"source": -1, "source_id": "G", "target_id": "G"}]
    },
    {
        "id": 1,
        "name": "find_max",
        "dependencies": [0],
        "inputs": [{"source": 0, "source_id": "PR", "target_id": "values"}]
    },
    {
        "id": 2,
        "name": "bfs",
        "dependencies": [0],
        "inputs": [{"source": -1, "source_id": "G", "target_id": "G"},
                   {"source": -1, "source_id": "source", "target_id": "source"}]
    },
    {
        "id": 3,
        "name": "find_path",
        "dependencies": [1,2],
        "inputs": [{"source": 2, "source_id": "parents", "target_id": "parents"},
                   {"source": 1, "source_id": "result", "target_id": "end"},
                   {"source": -1, "source_id": "source", "target_id": "start"}]
    }
]

hardware = {
    'hosts': [{
        'id': 1,
        'name': 'H01',
        'cpus': {
            'id': 1,
            'name': 'AMD Ryzen 7 PRO 6850H with Radeon Graphics',
            'clock_speed': 2.1,
            'cores': 8,
            'threads': 16,
            'wattage': 35,
            'amount': 1,
            'benchmarks': {
                'T_int_add': 1.696241,
                'T_int_mult': 0.268697,
                'T_int_gt': 0.1083579, 
                'T_int_neq': 0.2161579,
                'T_float_add': 0.4116942,
                'T_float_sub': 0.4075082,
                'T_float_mult': 0.4063832,
                'T_float_div': 0.9677004,
                'T_float_gt': 0.1120397,
                'T_q_push': 11.39776,
                'T_q_front': 8.72066,
                'T_q_pop': 8.735464,
                'T_heap_insert_max': 39.34497,
                'T_heap_extract_min': 75.7469,
                'T_heap_decrease_key': 7.046225,
                'T_push_back': 11.17706,
                'L1_linesize': 64,
                'L2_linesize': 64,
                'L3_linesize': 64,
                'T_L1_read': 2.0811750265498867,
                'T_L2_read': 5.106190276464619,
                'T_L3_read': 23.70627666007128,
                'T_DRAM_read': 110.75488153099849
            }
        },
        'gpus': {
            'name': 'NVIDIA T600 Laptop GPU',
            'memory': '4096 MiB',
            'wattage': '35.00 W',
            'compute_capability': '7.5',
            'driver_version': '590.48.01'
        }
    }]
}

graph_properties = {
    "name": "data/large/wikipedia_link_en.mtx",
    "graph_sample": "data/samples/wikipedia_link_en_sample_001.mtx",
    "n": 456290,
    "m": 12508244,
    "average_degree": 27.41292599,
    "directed": False,
    "weighted": False,
    "diameter": 9,
    "clustering_coefficient": 0.1887,
    "triangle_count": 83023401,
    "s": 1000
}

#### End-to-end prediction

In [ ]:
target = RUNTIME
correction = 2.2
predicted_runtime, predicted_energy, annotated_workflow, all_predictions = graph_optimizer_e2e(workflow, hardware, graph_properties, target, correction)

#### Automatic code generation

In [ ]:
code_dir = "code_generation/generated_code"
user_input = {"G": graph_properties["name"], "source": 1}
generate_code(annotated_workflow, user_input=user_input, host="H01", code_dir=code_dir, include_timing=True, print_output=True)

An execution plan is made for a single host

<img src="execution_plan.svg">

##### Compiling the generated code

In [ ]:
!(cd {code_dir} && make)

##### Running the generated code

In [ ]:
optimal_output = subprocess.check_output([f'{code_dir}/main']).decode('utf-8')
optimal_output

#### Prediction validation

In [ ]:
bars = {'Prediction': (predicted_runtime, predicted_energy),
        'Validation': convert_output(optimal_output)}

plot_runtimes(bars)

runtime_error, energy_error = get_error(bars)

print(f"The runtime error is {runtime_error*100}%")
print(f"The energy error is {energy_error*100}%")

#### How much did Graph-Optimizer save?

In [ ]:
_, _, worst_case_annotated_workflow = select_worst_case(workflow, all_predictions)
generate_code(worst_case_annotated_workflow, user_input=user_input, host="H01", code_dir=code_dir, include_timing=True)
!(cd {code_dir} && make)
worst_case_output = subprocess.check_output([f'{code_dir}/main']).decode('utf-8')
bars['Worst Case'] = convert_output(worst_case_output)
plot_runtimes(bars)

## The input: workflow, hardware and graph properties

#### Workflow is specified by the user

<img src="dag.svg">

In [ ]:
workflow = [
    {
        "id": 0,
        "name": "pr",
        "dependencies": [],
    },
    {
        "id": 1,
        "name": "find_max",
        "dependencies": [0],
    },
    {
        "id": 2,
        "name": "bfs",
        "dependencies": [0],
    },
    {
        "id": 3,
        "name": "find_path",
        "dependencies": [1,2],
    }
]

#### Hardware information  is automatically extracted

In [ ]:
name = 'demo_host'
power_draw = 45
include_gpu = True
hardware = extract_hardware_characteristics(name, power_draw, include_gpu)
hardware

Including microbenchmarks, more on this later

In [ ]:
hardware = hardware_and_microbenchmarks(name, power_draw, include_gpu)
hardware

In [ ]:
hardware = {'hosts': [hardware]}

#### Graph characteristics are also automatically extracted

In [ ]:
demo_graph_name = 'data/test_matrix_fully_connected.mtx'
demo_graph_properties = extract_graph_properties(graph_name)
demo_graph_properties

## How does graph-optimizer know which implementation is the best?

#### Analytical models

In [ ]:
analytical_annotated_workflow = analytical_prediction(hardware, workflow, graph_properties) 
interact(lambda target: plot_result(analytical_annotated_workflow, 'Analytical Performance Prediction', target), target=ToggleButtons(options=['runtime', 'energy'], description='Prediction target'));

#### Sampling based prediction

Based on:

Bart, D., Chen, K., Varbanescu, A.: *Millibenchmarking: Using Graph Sampling for Ranking GPU PageRank Implementations*. GraphSys 2025

In [ ]:
sampling_rate = 0.1
sampled_graph_name = 'sampled.mtx'
!./sampling/main {graph_name} {sampling_rate} {sampled_graph_name} && echo "Succesfully sampled the graph"
graph_properties["graph_sample"] = sampled_graph_name

In [ ]:
interact(lambda target: plot_result(all_predictions, 'Sampling Based Prediction Added', target), target=ToggleButtons(options=['runtime', 'energy'], description='Prediction target'));

## Automatic code generation

#### Workflow is specified by the user, with annotated inputs

<img src="dag_with_inputs.svg">

In [ ]:
workflow = [
    {
        "id": 0,
        "name": "pr",
        "dependencies": [],
        "inputs": [{"source": -1, "source_id": "G", "target_id": "G"}]
    },
    {
        "id": 1,
        "name": "find_max",
        "dependencies": [0],
        "inputs": [{"source": 0, "source_id": "PR", "target_id": "values"}]
    },
    {
        "id": 2,
        "name": "bfs",
        "dependencies": [0],
        "inputs": [{"source": -1, "source_id": "G", "target_id": "G"},
                   {"source": -1, "source_id": "source", "target_id": "source"}]
    },
    {
        "id": 3,
        "name": "find_path",
        "dependencies": [1,2],
        "inputs": [{"source": 2, "source_id": "parents", "target_id": "parents"},
                   {"source": 1, "source_id": "result", "target_id": "end"},
                   {"source": -1, "source_id": "source", "target_id": "start"}]
    }
]

#### Optimal implementations are predicted and inserted in the workflow
with the end-to-end prediction method

In [ ]:
annotated_workflow = [
    {
        'id': 0,
        'name': 'pr',
        'dependencies': [],
        'inputs': [{'source': -1, 'source_id': 'G', 'target_id': 'G'}],
        'performances': [
            {
                'host': 'H01',
                'implementation': 'CPU/pr_gap',
                'runtime': 31914804790.0,
                'energy': 803181110.0
            }
        ]
    },
    {
        'id': 1,
        'name': 'find_max',
        'dependencies': [0],
        'inputs': [{'source': 0, 'source_id': 'PR', 'target_id': 'values'}],
        'performances': [
            {
                'host': 'H01',
                'implementation': 'CPU/find_max_ca',
                'runtime': 2.8534564647104044,
                'energy': 10.02278935619039
            }
        ]
    },
    {
        'id': 2,
        'name': 'bfs',
        'dependencies': [0],
        'inputs': [{'source': -1, 'source_id': 'G', 'target_id': 'G'},
                   {'source': -1, 'source_id': 'source', 'target_id': 'source'}],
        'performances': [
            {
                'host': 'H01',
                'implementation': 'CPU/bfs_gap',
                'runtime': 1337710.0004947442,
                'energy': 0.04681985001731605
            }
        ]
    },
    {
        'id': 3,
        'name': 'find_path',
        'dependencies': [1, 2],
        'inputs': [{'source': 2, 'source_id': 'parents', 'target_id': 'parents'},
                   {'source': 1, 'source_id': 'result', 'target_id': 'end'},
                   {'source': -1, 'source_id': 'source', 'target_id': 'start'}],
        'performances': [
            {
                'host': 'H01',
                'implementation': 'CPU/fp_ca',
                'runtime': 0.00013304792451113682,
                'energy': 4.656677357889789e-12
            }
        ]
    }
]

#### User input is specified, and code and Makefile are generated

In [ ]:
user_input = {"G": graph_properties["name"], "source": 1}
generate_code(annotated_workflow, user_input)
!cat {code_dir}/main.cpp

## Synergy with Graph-Greenifier

#### Output from Graph-Optimizer can be used as input for Graph-Greenifier

In [ ]:
greenifier_input = convert_to_greenifier_input(annotated_workflow)
greenifier_input